<a href="https://colab.research.google.com/github/govardhansnair/binomial-option-pricing-framework/blob/main/Live_Binomial_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# QUANTITATIVE FINANCE TERMINAL

# DISCLAIMER- This model Auto-refreshes every 60 seconds and relies on yahoo finance data.
# Verify data using previous binomial model file.


# 1. INSTALL LIBRARIES

!pip -q install yfinance ipywidgets numpy pandas

import numpy as np
import pandas as pd
import yfinance as yf
import ipywidgets as widgets
import threading
import time

from datetime import date, timedelta
from IPython.display import display, clear_output


# 2. DOWNLOAD MARKET DATA


def get_market_data(ticker):

    ticker = ticker.strip().upper()

    stock = yf.Ticker(ticker)

    # Download one year of daily prices
    history = stock.history(
        period="1y",
        interval="1d",
        auto_adjust=True
    )

    if history.empty:
        raise ValueError("No historical data found.")

    prices = history["Close"].dropna()

    # Calculate logarithmic returns
    returns = np.log(prices / prices.shift(1)).dropna()

    # Calculate annualised volatility
    volatility = returns.std(ddof=1) * np.sqrt(252)

    if not np.isfinite(volatility) or volatility <= 0:
        raise ValueError("Invalid volatility.")

    # Download latest available intraday price
    intraday = stock.history(
        period="1d",
        interval="1m",
        auto_adjust=False
    )

    intraday_prices = (
        intraday["Close"].dropna()
        if not intraday.empty
        else pd.Series(dtype=float)
    )

    if not intraday_prices.empty:

        spot = float(intraday_prices.iloc[-1])

        timestamp = str(intraday_prices.index[-1])

        source = "Latest available 1-minute bar"

    else:

        spot = float(prices.iloc[-1])

        timestamp = str(prices.index[-1])

        source = "Latest available daily close"

    return {
        "ticker": ticker,
        "spot": spot,
        "volatility": float(volatility),
        "timestamp": timestamp,
        "source": source
    }



# 3. BINOMIAL OPTION PRICING MODEL

def price_option(
    spot,
    strike,
    rate,
    volatility,
    maturity,
    steps=200,
    option_type="call",
    exercise="european"
):

    if spot <= 0 or strike <= 0:
        raise ValueError("Spot and strike must be positive.")

    if volatility <= 0 or maturity <= 0:
        raise ValueError("Volatility and maturity must be positive.")

    dt = maturity / steps

    # Up and down factors
    u = np.exp(volatility * np.sqrt(dt))

    d = 1 / u

    # Risk-neutral probability
    q = (np.exp(rate * dt) - d) / (u - d)

    if q < 0 or q > 1:
        raise ValueError("Invalid risk-neutral probability.")

    discount = np.exp(-rate * dt)

    # Terminal stock prices
    j = np.arange(steps + 1)

    stock_prices = spot * (u ** j) * (d ** (steps - j))

    # Terminal option payoffs
    if option_type == "call":

        option_values = np.maximum(
            stock_prices - strike,
            0
        )

    else:

        option_values = np.maximum(
            strike - stock_prices,
            0
        )

    # Backward induction
    for level in range(steps - 1, -1, -1):

        continuation = discount * (
            q * option_values[1:level + 2]
            + (1 - q) * option_values[:level + 1]
        )

        if exercise == "american":

            j = np.arange(level + 1)

            stock_prices = spot * (
                u ** j
            ) * (
                d ** (level - j)
            )

            if option_type == "call":

                intrinsic = np.maximum(
                    stock_prices - strike,
                    0
                )

            else:

                intrinsic = np.maximum(
                    strike - stock_prices,
                    0
                )

            option_values = np.maximum(
                intrinsic,
                continuation
            )

        else:

            option_values = continuation

    return float(option_values[0])



# 4. GREEKS CALCULATOR


def calculate_greeks(
    spot,
    strike,
    rate,
    volatility,
    maturity,
    steps,
    option_type,
    exercise
):

    def calculate(s, v, t):

        return price_option(
            s,
            strike,
            rate,
            v,
            t,
            steps,
            option_type,
            exercise
        )

    base = calculate(spot, volatility, maturity)

    # Delta and Gamma
    bump = max(spot * 0.001, 0.0001)

    up = calculate(
        spot + bump,
        volatility,
        maturity
    )

    down = calculate(
        spot - bump,
        volatility,
        maturity
    )

    delta = (up - down) / (2 * bump)

    gamma = (up - 2 * base + down) / bump**2

    # Vega
    vol_bump = 0.01

    vega = (
        calculate(spot, volatility + vol_bump, maturity)
        -
        calculate(spot, max(volatility - vol_bump, 1e-8), maturity)
    ) / (2 * vol_bump)

    # Theta
    time_bump = min(1 / 365, maturity / 2)

    shorter = calculate(
        spot,
        volatility,
        maturity - time_bump
    )

    theta = (shorter - base) / (time_bump * 365)

    return {
        "price": base,
        "delta": delta,
        "gamma": gamma,
        "vega": vega,
        "vega_per_point": vega * 0.01,
        "theta": theta
    }



# 5. CREATE DASHBOARD INPUTS


ticker_input = widgets.Text(
    value="AAPL",
    description="Stock ticker:"
)

strike_input = widgets.FloatText(
    value=200,
    description="Strike price:"
)

rate_input = widgets.FloatText(
    value=4.0,
    description="Risk-free rate (%):"
)

expiry_input = widgets.DatePicker(
    description="Expiry date:",
    value=date.today() + timedelta(days=30)
)

option_input = widgets.Dropdown(
    options=["call", "put"],
    value="call",
    description="Option type:"
)

exercise_input = widgets.Dropdown(
    options=["european", "american"],
    value="european",
    description="Exercise:"
)

steps_input = widgets.IntSlider(
    value=200,
    min=25,
    max=500,
    step=25,
    description="Tree steps:"
)

auto_refresh = widgets.Checkbox(
    value=False,
    description="Auto refresh every 60 seconds"
)

refresh_button = widgets.Button(
    description="REFRESH MARKET DATA",
    button_style="success"
)

output = widgets.Output()



# 6. DASHBOARD CALCULATIONS


def update_dashboard():

    with output:

        clear_output(wait=True)

        try:

            ticker = ticker_input.value.strip().upper()

            strike = strike_input.value

            rate = rate_input.value / 100

            expiry = expiry_input.value

            option_type = option_input.value

            exercise = exercise_input.value

            steps = steps_input.value

            if expiry is None:
                raise ValueError("Select an expiry date.")

            days = (expiry - date.today()).days

            if days <= 0:
                raise ValueError("Expiry must be in the future.")

            maturity = days / 365

            # Download market data
            market = get_market_data(ticker)

            spot = market["spot"]

            volatility = market["volatility"]

            # Calculate option price
            option_price = price_option(
                spot,
                strike,
                rate,
                volatility,
                maturity,
                steps,
                option_type,
                exercise
            )

            # Calculate Greeks
            greeks = calculate_greeks(
                spot,
                strike,
                rate,
                volatility,
                maturity,
                steps,
                option_type,
                exercise
            )

            clear_output(wait=True)

            print("=" * 55)

            print("       QUANTITATIVE FINANCE TERMINAL")

            print("=" * 55)

            print("\nMARKET INFORMATION")

            print("-" * 55)

            print("Ticker:", ticker)

            print("Latest available stock price:", round(spot, 4))

            print("Annualised volatility:",
                  round(volatility * 100, 2), "%")

            print("Price source:", market["source"])

            print("Price timestamp:", market["timestamp"])

            print("\nOPTION INFORMATION")

            print("-" * 55)

            print("Strike:", strike)

            print("Risk-free rate:", rate * 100, "%")

            print("Days to expiry:", days)

            print("Option type:", option_type.upper())

            print("Exercise style:", exercise.upper())

            print("Tree steps:", steps)

            print("\nOPTION PRICING RESULTS")

            print("-" * 55)

            print("Binomial option price:", round(option_price, 4))

            print("\nOPTION GREEKS")

            print("-" * 55)

            print("Delta:", round(greeks["delta"], 6))

            print("Gamma:", round(greeks["gamma"], 6))

            print("Vega:", round(greeks["vega"], 6))

            print("Vega per volatility point:",
                  round(greeks["vega_per_point"], 6))

            print("Theta per day:", round(greeks["theta"], 6))

            print("\n" + "=" * 55)

            print("Data may be delayed and is not guaranteed real-time.")

            print("The option price is theoretical, not an exchange quote.")

            print("=" * 55)

        except Exception as error:

            clear_output(wait=True)

            print("ERROR:", error)



# 7. REFRESH BUTTON


def refresh_clicked(button):

    update_dashboard()


refresh_button.on_click(refresh_clicked)


# 8. AUTOMATIC REFRESH


def automatic_refresh():

    while True:

        time.sleep(60)

        if auto_refresh.value:

            update_dashboard()


thread = threading.Thread(
    target=automatic_refresh,
    daemon=True
)

thread.start()



# 9. DISPLAY DASHBOARD


display(

    widgets.VBox([

        widgets.HTML(
            "<h2>Quantitative Finance Terminal</h2>"
            "<p>Market-linked Binomial Option Pricing System</p>"
        ),

        ticker_input,

        strike_input,

        rate_input,

        expiry_input,

        option_input,

        exercise_input,

        steps_input,

        auto_refresh,

        refresh_button,

        output

    ])

)


# Run dashboard once
update_dashboard()